In [8]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.pytorch
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [27]:
df = pd.read_csv('../data/data_preprocessed.csv')
mlflow.pytorch.autolog()
df['returns'] = np.log(df['CLOSE'] / df['CLOSE'].shift(1))
df.dropna(inplace=True)
df

,date,views,number_of_news,sentiment_score,CLOSE,VOLUME,returns
1,2006-07-20,501,17,4,203.26,1141288.0,-0.003389
2,2006-07-21,1038,22,3,202.00,1864106.0,-0.006218
3,2006-07-24,553,17,8,201.99,794152.0,-0.000050
4,2006-07-25,756,27,7,202.69,1674834.0,0.003460
5,2006-07-26,1287,30,4,203.10,1009200.0,0.002021
...,...,...,...,...,...,...,...
4857,2025-12-15,168207,50,5,409.65,3688415.0,0.018229
4858,2025-12-16,70523,37,9,413.00,3507863.0,0.008144
4859,2025-12-17,129196,47,6,411.45,2995120.0,-0.003760
4860,2025-12-18,152050,50,2,409.25,5504714.0,-0.005361


In [25]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, step, target_col, feature_col, scaler_features, scaler_target):
        self.seq_len = seq_len
        self.step = step

        self.features = data[feature_col].values.astype(np.float32)
        self.target = data[target_col].values.astype(np.float32).reshape(-1, 1)

        # scaler 
        self.scaler_features = scaler_features
        self.scaler_target = scaler_target

        self.features = self.scaler_features.transform(self.features)
        self.target = self.scaler_target.transform(self.target) 

        #self.feature_means = np.mean(self.features, axis=0)
        #self.feature_vars = np.var(self.features, axis=0)

    
    def __len__(self):
        return (len(self.features) - self.seq_len) // self.step
    
    
    def __getitem__(self, idx):
        start_idx = idx * self.step
        end_idx = start_idx + self.seq_len

        x = self.features[start_idx:end_idx]
        y = self.target[end_idx]

        return torch.from_numpy(x), torch.from_numpy(y)

In [51]:
# Параметры для настройки
#   -   #   -   #   -   #   -   #   -   #
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:]

seq_len = 10
step = 1
target_col = ['returns']
feature_col = ['sentiment_score', 'VOLUME']
scaler_features = StandardScaler().fit(train_df[feature_col])
scaler_target = StandardScaler().fit(train_df[target_col])

batch_size = 32
#   -   #   -   #   -   #   -   #   -   #

train_ds = TimeSeriesDataset(data=train_df,
                             seq_len=seq_len,
                             step=step,
                             target_col=target_col,
                             feature_col=feature_col,
                             scaler_features=scaler_features,
                             scaler_target=scaler_target)

val_ds = TimeSeriesDataset(data=val_df,
                           seq_len=seq_len,
                           step=step,
                           target_col=target_col,
                           feature_col=feature_col,
                           scaler_features=scaler_features,
                           scaler_target=scaler_target)


train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [52]:
class GRUPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, dropout=0.2):
        super(GRUPredictor, self).__init__()
        
        self.gru = nn.GRU(
            input_size=input_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_len, input_dim)
        # out: (batch_size, seq_len, hidden_dim)
        out, _ = self.gru(x)
        
        out = self.fc(out[:, -1, :]) 
        return out

In [53]:
def train(model, train_loader, val_loader, epochs, lr, weight_decay):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(device), y_val.to(device)
                preds = model(x_val)
                val_loss += criterion(preds, y_val).item()
        
        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.6f} | Val Loss: {val_loss/len(val_loader):.6f}')

In [54]:
model = GRUPredictor(input_dim=len(feature_col), hidden_dim=32, num_layers=2, output_dim=1)

train(model=model, 
      train_loader=train_loader, 
      val_loader=val_loader, 
      epochs=50, 
      lr=1e-4,
      weight_decay=1e-5)

Epoch 1/50 | Train Loss: 1.006662 | Val Loss: 1.142404
Epoch 2/50 | Train Loss: 1.005311 | Val Loss: 1.142073
Epoch 3/50 | Train Loss: 1.006165 | Val Loss: 1.142026
Epoch 4/50 | Train Loss: 1.006104 | Val Loss: 1.142010
Epoch 5/50 | Train Loss: 1.004592 | Val Loss: 1.141853
Epoch 6/50 | Train Loss: 1.006336 | Val Loss: 1.141714
Epoch 7/50 | Train Loss: 1.005023 | Val Loss: 1.141744
Epoch 8/50 | Train Loss: 1.004756 | Val Loss: 1.141596
Epoch 9/50 | Train Loss: 1.005065 | Val Loss: 1.141498
Epoch 10/50 | Train Loss: 1.004431 | Val Loss: 1.141558
Epoch 11/50 | Train Loss: 1.005361 | Val Loss: 1.141531
Epoch 12/50 | Train Loss: 1.004344 | Val Loss: 1.141571
Epoch 13/50 | Train Loss: 1.004923 | Val Loss: 1.141491
Epoch 14/50 | Train Loss: 1.004650 | Val Loss: 1.141444
Epoch 15/50 | Train Loss: 1.004858 | Val Loss: 1.141310
Epoch 16/50 | Train Loss: 1.005025 | Val Loss: 1.141272
Epoch 17/50 | Train Loss: 1.004334 | Val Loss: 1.141276


KeyboardInterrupt: 